# Sistema RAG Completo com Avaliação de Qualidade

## Prática 02 - Implementação Didática de RAG com Métricas

**Autor:** Jefferson Cunha  
**Contexto:** Doutorado em Ciência de Dados  
**Objetivo:** Implementar um sistema RAG (Retrieval-Augmented Generation) completo utilizando dados reais do HuggingFace, com foco em avaliação de qualidade através de métricas estabelecidas pela literatura.

---

## Visão Geral do Projeto

Este notebook implementa um sistema RAG end-to-end, desde a ingestão de dados até a avaliação com métricas rigorosas. O projeto é **didático e reprodutível**, priorizando a clareza conceitual sobre otimizações de produção.

### Por que RAG é importante?

Em domínios especializados como medicina e ciência, os LLMs generalistas enfrentam dois problemas críticos:

1. **Conhecimento desatualizado:** Modelos são treinados em dados históricos e não conhecem informações recentes
2. **Alucinações:** Tendem a gerar respostas plausíveis mas incorretas quando não possuem conhecimento factual

**RAG resolve esses problemas** ao combinar:
- **Retrieval:** Busca semântica em base de conhecimento atualizada
- **Augmentation:** Injeção do conhecimento recuperado no contexto do LLM
- **Generation:** Geração de resposta fundamentada nas evidências fornecidas

### Arquitetura do Sistema

```
┌─────────────┐
│   Query     │ "What is the role of p53 in cancer?"
└──────┬──────┘
       │
       ▼
┌─────────────────────────────────────────────────┐
│  1. INDEXAÇÃO (Offline)                         │
│  ┌──────────────┐   ┌──────────────┐            │
│  │ Documents    │──▶│  Chunking    │            │
│  │ Pool (740MB) │   │ (Overlap)    │            │
│  └──────────────┘   └──────┬───────┘            │
│                            │                     │
│                            ▼                     │
│                     ┌──────────────┐             │
│                     │  Embeddings  │             │
│                     │ (OpenAI API) │             │
│                     └──────┬───────┘             │
│                            │                     │
│                            ▼                     │
│                     ┌──────────────┐             │
│                     │  Vector DB   │             │
│                     │  (ChromaDB)  │             │
│                     └──────────────┘             │
└─────────────────────────────────────────────────┘
       │
       ▼
┌─────────────────────────────────────────────────┐
│  2. RETRIEVAL (Online)                          │
│                                                  │
│  Query Embedding ──▶ Similarity Search ──▶ Top-K│
│                      (Cosine Distance)    Docs  │
└────────────────────────────┬────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────┐
│  3. GENERATION                                  │
│                                                  │
│  ┌────────────┐   ┌────────────┐   ┌─────────┐ │
│  │   Query    │ + │ Retrieved  │──▶│   LLM   │ │
│  │            │   │  Context   │   │ (GPT-4) │ │
│  └────────────┘   └────────────┘   └────┬────┘ │
│                                         │       │
│                                         ▼       │
│                                    ┌─────────┐  │
│                                    │ Answer  │  │
│                                    └─────────┘  │
└─────────────────────────────────────────────────┘
       │
       ▼
┌─────────────────────────────────────────────────┐
│  4. AVALIAÇÃO                                   │
│                                                  │
│  ┌─────────────────┐  ┌──────────────────────┐ │
│  │ LLM Metrics     │  │ Retrieval Metrics    │ │
│  │ - Correctness   │  │ - Precision@K        │ │
│  │ - Faithfulness  │  │ - Recall@K           │ │
│  │                 │  │ - Contextual Metrics │ │
│  └─────────────────┘  └──────────────────────┘ │
└─────────────────────────────────────────────────┘
```

### Dataset: AQ-MedAI/RAG-QA-Leaderboard

Utilizaremos dados médicos/científicos do PubMed com estrutura completa para avaliação:

- **Queries:** Perguntas médicas/científicas
- **Ground Truth:** Respostas corretas esperadas
- **Golden Documents:** Documentos que contêm evidências para as respostas
- **Document Pool:** Corpus de 740MB com conhecimento da Wikipédia

### Métricas de Avaliação

**Geração (LLM-as-a-judge):**
- **Correctness (Answer Relevancy):** A resposta está completa e relevante?
- **Faithfulness:** A resposta está fundamentada no contexto ou alucinou?

**Recuperação (Information Retrieval):**
- **Precision@K:** Proporção de documentos relevantes recuperados
- **Recall@K:** Cobertura dos documentos golden
- **Contextual Recall/Precision/Relevancy:** Métricas contextuais avançadas

### Ferramentas Utilizadas

- **LangChain:** Orquestração do pipeline RAG
- **ChromaDB:** Vector database local
- **OpenAI:** Embeddings (text-embedding-3-small) e LLM (gpt-4o-mini)
- **DeepEval:** Framework de avaliação com LLM-as-a-judge
- **Langfuse:** Observabilidade e rastreamento de custos/latência

---

## Estrutura deste Notebook

1. **Configuração do Ambiente** - Setup de dependências e credenciais
2. **Aquisição de Dados** - Download e exploração do dataset HuggingFace
3. **Indexação** - Chunking, embeddings e criação do vector database
4. **Retriever** - Implementação de busca semântica
5. **Generator** - Geração de respostas com prompt engineering
6. **Avaliação LLM** - Métricas de correctness e faithfulness
7. **Avaliação IR** - Métricas de recuperação de informação
8. **Observabilidade** - Análise de custos e latência com Langfuse
9. **Casos Extremos** - Testes de edge cases e limitações
10. **Conclusões** - Resultados, aprendizados e próximos passos

---

# 1. Configuração do Ambiente

Nesta seção, configuraremos todas as dependências necessárias e validaremos a conectividade com as APIs externas (OpenAI e Langfuse).

## Dependências do Projeto

- `langchain` + `langchain-openai`: Framework para orquestração de LLMs
- `langchain-community`: Componentes comunitários (ChromaDB integration)
- `chromadb`: Vector database local para armazenamento de embeddings
- `openai`: Cliente oficial da OpenAI para embeddings e chat
- `langfuse`: Plataforma de observabilidade para LLMOps
- `deepeval`: Framework de avaliação com métricas LLM-as-a-judge
- `huggingface_hub`: Download de datasets do HuggingFace
- `pandas`, `matplotlib`, `seaborn`: Análise e visualização de dados
- `python-dotenv`: Gestão segura de variáveis de ambiente

In [ ]:
# Instalação de dependências (executar apenas uma vez)
# Descomente as linhas abaixo se precisar instalar os pacotes

# !pip install -q langchain langchain-openai langchain-community
# !pip install -q chromadb openai
# !pip install -q langfuse deepeval
# !pip install -q huggingface_hub datasets
# !pip install -q pandas matplotlib seaborn
# !pip install -q python-dotenv

print("✅ Instalação de dependências configurada. Descomente as linhas acima se necessário.")

## Configuração de Credenciais

Para executar este notebook, você precisa configurar as seguintes credenciais no arquivo `.env` na raiz do projeto:

```bash
OPENAI_API_KEY="sua-chave-openai"
LANGFUSE_PUBLIC_KEY="sua-chave-publica-langfuse"
LANGFUSE_SECRET_KEY="sua-chave-secreta-langfuse"
LANGFUSE_HOST="https://cloud.langfuse.com"
```

**Como obter as credenciais:**

1. **OpenAI API Key:** 
   - Acesse https://platform.openai.com/api-keys
   - Crie uma nova API key
   - Tenha créditos disponíveis na conta

2. **Langfuse Keys:**
   - Acesse https://cloud.langfuse.com/
   - Crie uma conta gratuita
   - Vá em Settings → API Keys
   - Copie Public Key, Secret Key e Host

Use o arquivo `.env.example` como template.

In [ ]:
import os
from dotenv import load_dotenv

# Carrega variáveis de ambiente do arquivo .env
load_dotenv()

# Validação de credenciais
required_vars = [
    "OPENAI_API_KEY",
    "LANGFUSE_PUBLIC_KEY",
    "LANGFUSE_SECRET_KEY",
    "LANGFUSE_HOST"
]

missing_vars = []
for var in required_vars:
    if not os.getenv(var):
        missing_vars.append(var)
        print(f"❌ {var} não configurada")
    else:
        # Mostra apenas os primeiros caracteres para segurança
        value = os.getenv(var)
        if "KEY" in var:
            masked = value[:8] + "..." if len(value) > 8 else "***"
            print(f"✅ {var}: {masked}")
        else:
            print(f"✅ {var}: {value}")

if missing_vars:
    print(f"\n⚠️  ATENÇÃO: {len(missing_vars)} variável(is) faltando no arquivo .env")
    print("Por favor, configure as credenciais antes de prosseguir.")
else:
    print("\n✅ Todas as credenciais configuradas com sucesso!")

## Teste de Conectividade

Vamos validar que conseguimos nos conectar às APIs necessárias.

In [ ]:
from openai import OpenAI
from langfuse import get_client

print("🔍 Testando conectividade com APIs...\n")

# Teste OpenAI
try:
    client_openai = OpenAI()
    # Teste simples com embedding
    response = client_openai.embeddings.create(
        model="text-embedding-3-small",
        input="teste de conectividade"
    )
    print(f"✅ OpenAI API: Conectado com sucesso")
    print(f"   Modelo: text-embedding-3-small")
    print(f"   Dimensões do embedding: {len(response.data[0].embedding)}")
except Exception as e:
    print(f"❌ OpenAI API: Falha na conexão")
    print(f"   Erro: {str(e)}")

print()

# Teste Langfuse
try:
    langfuse_client = get_client()
    print(f"✅ Langfuse: Cliente inicializado")
    print(f"   Host: {os.getenv('LANGFUSE_HOST')}")
    print(f"   Status: Pronto para rastreamento")
except Exception as e:
    print(f"❌ Langfuse: Falha na inicialização")
    print(f"   Erro: {str(e)}")

print("\n✅ Testes de conectividade concluídos!")

---

# 2. Aquisição e Exploração de Dados

## Dataset: AQ-MedAI/RAG-QA-Leaderboard

Este dataset foi criado especificamente para avaliação de sistemas RAG e contém:

### Arquivos Principais

1. **pubmed.jsonl** (583 KB)
   - Perguntas e respostas do domínio médico/científico
   - Cada linha contém um objeto JSON com: `id`, `query`, `golden_doc`, `reference`, `ground_truth`
   
2. **documents_pool.json** (740 MB)
   - Corpus centralizado de documentos da Wikipédia
   - Estrutura: `{document_id: {title, text, url}}`
   - Os IDs em `golden_doc` e `reference` referenciam documentos neste pool

### Estrutura dos Dados

**Exemplo de registro em pubmed.jsonl:**
```json
{
  "id": "pubmed_001",
  "query": "What is the role of p53 in cancer?",
  "golden_doc": ["Document_12345", "Document_67890"],
  "reference": ["Document_11111", "Document_22222"],
  "ground_truth": ["p53 is a tumor suppressor..."]
}
```

**Campos:**
- `query`: Pergunta médica/científica
- `golden_doc`: IDs dos documentos "ouro" que contêm evidências para a resposta
- `reference`: IDs de documentos adicionais relacionados
- `ground_truth`: Resposta(s) correta(s) esperada(s)

### Estratégia de Amostragem

Para fins didáticos, trabalharemos com um **subset otimizado**:
1. Carregar todas as queries do pubmed.jsonl
2. Extrair apenas os documentos referenciados (em vez dos 740MB completos)
3. Usar amostra de 50-100 queries para desenvolvimento
4. Manter código escalável para dataset completo

## 2.1. Download dos Dados do HuggingFace

Vamos baixar os arquivos necessários usando a biblioteca `huggingface_hub`.

In [ ]:
from huggingface_hub import hf_hub_download
import json
from pathlib import Path

print("📥 Iniciando download dos dados do HuggingFace...\n")

# Configurações
REPO_ID = "AQ-MedAI/RAG-QA-Leaderboard"
CACHE_DIR = "./data"  # Diretório local para cache

# Criar diretório se não existir
Path(CACHE_DIR).mkdir(exist_ok=True)

# Download pubmed.jsonl
print("1️⃣  Baixando pubmed.jsonl...")
try:
    pubmed_path = hf_hub_download(
        repo_id=REPO_ID,
        filename="final_data/pubmed.jsonl",
        repo_type="dataset",
        cache_dir=CACHE_DIR
    )
    print(f"   ✅ pubmed.jsonl baixado: {pubmed_path}")
except Exception as e:
    print(f"   ❌ Erro ao baixar pubmed.jsonl: {e}")
    pubmed_path = None

print()

# Download documents_pool.json
print("2️⃣  Baixando documents_pool.json (740 MB - pode demorar alguns minutos)...")
try:
    docs_pool_path = hf_hub_download(
        repo_id=REPO_ID,
        filename="final_data/documents_pool.json",
        repo_type="dataset",
        cache_dir=CACHE_DIR
    )
    print(f"   ✅ documents_pool.json baixado: {docs_pool_path}")
except Exception as e:
    print(f"   ❌ Erro ao baixar documents_pool.json: {e}")
    docs_pool_path = None

print("\n✅ Download concluído!")

## 2.2. Carregamento e Parsing dos Dados

Vamos carregar os arquivos JSON/JSONL e explorar sua estrutura.

In [ ]:
import pandas as pd

print("📊 Carregando e parsing dos dados...\n")

# Carregar pubmed.jsonl (JSON Lines - uma query por linha)
print("1️⃣ Carregando pubmed.jsonl...")
pubmed_queries = []
with open(pubmed_path, 'r', encoding='utf-8') as f:
    for line in f:
        pubmed_queries.append(json.loads(line.strip()))

print(f"   ✅ {len(pubmed_queries)} queries carregadas")

# Mostrar exemplo de query
print("\n📋 Exemplo de query:")
print(json.dumps(pubmed_queries[0], indent=2))

print()

# Carregar documents_pool.json
print("2️⃣ Carregando documents_pool.json (pode demorar um pouco)...")
with open(docs_pool_path, 'r', encoding='utf-8') as f:
    documents_pool = json.load(f)

print(f"   ✅ {len(documents_pool):,} documentos carregados no pool")

# Mostrar exemplo de documento
doc_id = list(documents_pool.keys())[0]
print(f"\n📄 Exemplo de documento (ID: {doc_id}):")
print(json.dumps(documents_pool[doc_id], indent=2)[:200] + "...")

## 2.3. Análise Exploratória dos Dados

Vamos analisar as características do dataset para entender melhor sua estrutura.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar estilo dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 4)

# Análise das queries
df_queries = pd.DataFrame(pubmed_queries)

print("="*80)
print("ANÁLISE DO DATASET PUBMED")
print("="*80)

print(f"\n📊 Estatísticas Gerais:")
print(f"   Total de queries: {len(df_queries)}")
print(f"   Total de documentos no pool: {len(documents_pool):,}")

# Analisar campos
print(f"\n📋 Campos disponíveis:")
for col in df_queries.columns:
    print(f"   - {col}: {df_queries[col].dtype}")

# Estatísticas de golden_doc e reference
df_queries['num_golden'] = df_queries['golden_doc'].apply(len)
df_queries['num_reference'] = df_queries['reference'].apply(len)

print(f"\n📚 Documentos por Query:")
print(f"   Golden docs: {df_queries['num_golden'].mean():.2f} (média), min={df_queries['num_golden'].min()}, max={df_queries['num_golden'].max()}")
print(f"   Reference docs: {df_queries['num_reference'].mean():.2f} (média), min={df_queries['num_reference'].min()}, max={df_queries['num_reference'].max()}")

# Tamanhos de texto
df_queries['query_length'] = df_queries['query'].str.len()
df_queries['gt_length'] = df_queries['ground_truth'].apply(lambda x: len(str(x)))

print(f"\n📏 Tamanho dos Textos:")
print(f"   Queries: {df_queries['query_length'].mean():.0f} chars (média)")
print(f"   Ground truth: {df_queries['gt_length'].mean():.0f} chars (média)")

# Visualizações
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuição de tamanho das queries
axes[0].hist(df_queries['query_length'], bins=30, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Tamanho da Query (caracteres)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição: Tamanho das Queries')
axes[0].grid(True, alpha=0.3)

# Distribuição de golden docs
axes[1].hist(df_queries['num_golden'], bins=range(df_queries['num_golden'].max()+2), color='lightcoral', edgecolor='black')
axes[1].set_xlabel('Número de Golden Docs')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição: Golden Documents por Query')
axes[1].grid(True, alpha=0.3)

# Distribuição de reference docs
axes[2].hist(df_queries['num_reference'], bins=30, color='lightgreen', edgecolor='black')
axes[2].set_xlabel('Número de Reference Docs')
axes[2].set_ylabel('Frequência')
axes[2].set_title('Distribuição: Reference Documents por Query')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)

## 2.4. Criação de Subset para Desenvolvimento

Para fins didáticos, vamos criar um subset otimizado contendo apenas os documentos referenciados pelas queries do pubmed e uma amostra de queries para trabalhar.

In [ ]:
# Configuração: número de queries para amostra
SAMPLE_SIZE = 50  # Ajuste conforme necessário (50-100 para fins didáticos)

print("🔨 Criando subset otimizado...\n")

# 1. Selecionar amostra de queries
print(f"1️⃣ Selecionando {SAMPLE_SIZE} queries do dataset...")
sample_queries = pubmed_queries[:SAMPLE_SIZE]
print(f"   ✅ {len(sample_queries)} queries selecionadas")

# 2. Coletar todos os document IDs referenciados
print("\n2️⃣ Coletando document IDs referenciados...")
all_doc_ids = set()
for query in sample_queries:
    all_doc_ids.update(query['golden_doc'])
    all_doc_ids.update(query['reference'])

print(f"   ✅ {len(all_doc_ids)} document IDs únicos encontrados")

# 3. Extrair subset do documents_pool
print("\n3️⃣ Extraindo subset de documentos...")
subset_documents = {}
missing_docs = []

for doc_id in all_doc_ids:
    if doc_id in documents_pool:
        subset_documents[doc_id] = documents_pool[doc_id]
    else:
        missing_docs.append(doc_id)

print(f"   ✅ {len(subset_documents)} documentos extraídos")
if missing_docs:
    print(f"   ⚠️  {len(missing_docs)} documentos não encontrados no pool")

# 4. Estatísticas do subset
print("\n📊 Estatísticas do Subset:")
print(f"   Queries: {len(sample_queries)}")
print(f"   Documentos: {len(subset_documents)}")

# Calcular tamanho total do texto
total_chars = sum(len(doc.get('text', '')) for doc in subset_documents.values())
total_mb = total_chars / (1024 * 1024)
print(f"   Tamanho total do corpus: {total_chars:,} caracteres ({total_mb:.2f} MB)")
print(f"   Tamanho médio por documento: {total_chars // len(subset_documents):,} caracteres")

# 5. Validação de integridade
print("\n✅ Validação:")
coverage = (len(subset_documents) / len(all_doc_ids)) * 100
print(f"   Cobertura de documentos: {coverage:.1f}%")
print(f"   Dataset otimizado criado com sucesso!")

print("\n" + "="*80)

---

# Próximos Passos

Esta primeira versão do notebook implementou as etapas iniciais do sistema RAG:

## ✅ Implementado

1. **Introdução e Contexto** - Explicação completa da arquitetura RAG e objetivos do projeto
2. **Configuração do Ambiente** - Setup de dependências, credenciais e validação de conectividade
3. **Aquisição de Dados** - Download do dataset HuggingFace (pubmed.jsonl + documents_pool.json)
4. **Análise Exploratória** - Estatísticas e visualizações do dataset
5. **Criação de Subset** - Otimização do dataset para fins didáticos

## 🚧 Próximas Implementações

As seguintes seções serão implementadas nas próximas iterações:

### 3. Indexação dos Documentos (Passo 1 da Especificação)
- Implementar chunking com RecursiveCharacterTextSplitter
- Gerar embeddings com OpenAI (text-embedding-3-small)
- Criar e popular ChromaDB com metadados preservados
- Validar índice criado

### 4. Retriever (Passo 2 da Especificação)
- Implementar busca semântica com Langfuse tracing
- Testar com diferentes valores de k
- Analisar qualidade dos documentos recuperados

### 5. Generator (Passo 3 da Especificação)
- Criar prompt template restritivo
- Implementar geração com LLM
- Instrumentar com Langfuse para observabilidade
- Pipeline RAG completo

### 6. Avaliação - Métricas LLM (Passos 4 e 5 da Especificação)
- Implementar Answer Relevancy (Correctness) com DeepEval
- Implementar Faithfulness com DeepEval
- Visualizar resultados em tabelas e gráficos

### 7. Avaliação - Métricas de Recuperação (Passo 6 - Bônus)
- Implementar Precision@K e Recall@K clássicos
- Implementar Contextual Recall, Precision e Relevancy
- Análise comparativa com diferentes configurações

### 8. Observabilidade e Casos Extremos
- Integração completa com Langfuse
- Análise de custos e latências
- Testes de edge cases
- Documentação de limitações

## 📚 Referências

- **Dataset:** [AQ-MedAI/RAG-QA-Leaderboard](https://huggingface.co/datasets/AQ-MedAI/RAG-QA-Leaderboard)
- **RAG Benchmarks:** https://www.evidentlyai.com/blog/rag-benchmarks
- **LangChain Documentation:** https://python.langchain.com/
- **DeepEval Documentation:** https://docs.confident-ai.com/
- **Langfuse Documentation:** https://langfuse.com/docs

---

**Status:** Notebook em desenvolvimento - Seções 1-3 implementadas  
**Próxima atualização:** Seções 4-6 (Indexação, Retriever, Generator)